In [ ]:
# Установим все нужные библиотеки для проекта
!pip install langchain==0.0.330 sentence-transformers faiss-cpu docx2txt python-telegram-bot --upgrade

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.5/669.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.5 MB/s eta 0:00:00
   ━━━━

In [ ]:
import torch
if torch.cuda.is_available():
    print('=========== GPU подключен! ===========')
    # !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python==0.2.85
    # !CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python[server]==0.2.85
    !pip install llama-cpp-python[server]==0.2.85

else:
    print('=========== CPU подключен! ===========')
    !pip install -q --upgrade --force-reinstall llama-cpp-python==0.2.85 --no-cache-dir

=========== GPU подключен! ===========
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 MB 15.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 7.3 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.2.85-cp311-cp311-linux_x86_64.whl size=2873169 sha256=d186723b4e29fce2258d7adb06323cf11f5b66fa7242f1851e7ea65a9c7e8a25
  Stored in directory: /root/.cache/pip/wheels/15/49/90/42743b59290803ffaf9f60b8439239225e181ae2faaa83640c
Successfully built 

In [ ]:
# Импортируем прогресс-бар для Jupyter Notebook
from tqdm import tqdm_notebook as tqdm

# Импортируем класс Document из langchain для работы с документами
from langchain.document_loaders import UnstructuredFileLoader

In [ ]:
# Импорт необходимых библиотек
import os
import re
import copy
import time
import pickle
import requests
import textwrap
import logging
import warnings
from tqdm import tqdm
from telegram import Update, ReplyKeyboardMarkup
from telegram.ext import Application, CommandHandler, MessageHandler, filters, CallbackContext
from langchain.document_loaders import Docx2txtLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from llama_cpp import Llama

# Игнорирование предупреждений и логов
warnings.filterwarnings("ignore")
logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s", level=logging.INFO)
logging.getLogger("langchain.text_splitter").setLevel(logging.ERROR)

In [ ]:
# Функция для загрузки и разбиения текста из .docx документа
def load_and_split_text(docx_file, chunk_size=512, chunk_overlap=50):
    print(f"Загружаем файл {docx_file}...")
    try:
        loader = Docx2txtLoader(docx_file)
        documents = loader.load()
        print(f"Файл успешно загружен. Всего {len(documents)} документов.")
    except Exception as e:
        print(f"Ошибка при загрузке файла {docx_file}: {e}")
        return []

    print("Разбиваем текст на части...")
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    split_docs = splitter.split_documents(documents)
    print(f"Текст разбит на {len(split_docs)} частей.")
    return split_docs

In [ ]:
# Функция генерации FAISS индекса
def create_faiss_index_from_docs(docs, model_name="intfloat/multilingual-e5-large-instruct"):
    print("Генерируем эмбеддинги и создаём FAISS индекс...")
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    faiss_index = FAISS.from_documents(docs, embeddings)
    print("FAISS индекс успешно создан.")
    return faiss_index

# Функция сохранения FAISS индекса
def save_faiss_index(index, file_path):
    with open(file_path, 'wb') as f:
        pickle.dump(index, f)
    print(f"FAISS индекс успешно сохранён в файл {file_path}.")

In [ ]:
# Путь к медицинскому документу и файлу индекса
docx_file_path = "/content/diagnoses_symptoms_selenium.docx"
pickle_file_path = "/content/faiss_index.pkl"
model_name = "intfloat/multilingual-e5-large-instruct"  # Модель эмбеддингов

In [ ]:
# Шаг 1: Загрузка и разбиение текста
documents = load_and_split_text(docx_file_path)

# Шаг 2: Создание FAISS индекса
faiss_index = create_faiss_index_from_docs(documents, model_name=model_name)

# Шаг 3: Сохранение FAISS индекса
save_faiss_index(faiss_index, pickle_file_path)

Загружаем файл /content/diagnoses_symptoms_selenium.docx...
Файл успешно загружен. Всего 1 документов.
Разбиваем текст на части...
Текст разбит на 3893 частей.
Генерируем эмбеддинги и создаём FAISS индекс...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS индекс успешно создан.
FAISS индекс успешно сохранён в файл /content/faiss_index.pkl.


In [ ]:
# Конфигурация модели ruadapt_qwen2.5_7B
model_config =  {
      "url":'https://huggingface.co/RefalMachine/ruadapt_qwen2.5_7B_ext_u48_instruct_gguf/resolve/main/Q8_0.gguf',
      "chat_format": None,  # Формат сообщений (None означает стандартный формат)
      "n_ctx": 4096,  # Максимальная длина контекста для модели
      "n_gpu_layers": 41  # Количество слоев, которые будут выполняться на GPU
}

In [ ]:
# Функция скачивания модели
def download_model(model_config):
    url = model_config.get('url')
    filename = url.split('/')[-1]

    if not os.path.exists(filename):
        response = requests.get(url, stream=True)
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024
        t = tqdm(total=total_size, unit='iB', unit_scale=True)
        with open(filename, 'wb') as file:
            for data in response.iter_content(block_size):
                t.update(len(data))
                file.write(data)
        t.close()
        if total_size != 0 and t.n != total_size:
            raise ValueError("Ошибка при загрузке файла модели.")
    else:
        print(f"Файл модели {filename} уже существует.")
    return filename

# Функция загрузки модели Llama
def load_llama_model(filename, model_config):
    llm = Llama(
        model_path=filename,
        chat_format=model_config['chat_format'],
        n_ctx=model_config['n_ctx'],
        n_threads=16,
        n_gpu_layers=model_config['n_gpu_layers'],
        f16_kv=True,
        verbose=False
    )
    return llm

In [ ]:
# Скачивание и загрузка модели
filename = download_model(model_config)
llm = load_llama_model(filename, model_config)

100%|██████████| 8.06G/8.06G [03:12<00:00, 41.9MiB/s]


In [ ]:
def answer_kons_llama(model, system, instruction, topic, search_index, summary_history, temp=1, verbose=0, k=3, max_tokens=1000):
    """
    Генерирует уточняющий медицинский вопрос на основе контекста и базы знаний.
    """
    # Поиск документов, похожих на заданный симптом
    docs = search_index.similarity_search(topic, k=k)

    # Формируем контекст из найденных документов
    message_content = re.sub(r'\r\n', ' ', '\n '.join([f'\n--------------------\n' + doc.page_content + '\n' for doc in docs]))

    if verbose:
        print('Выбранные чанки:\n=========================================\n', message_content)

    # Формируем промпт для модели
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"{instruction}.\n{message_content}.\n\nВопрос:\n{topic}\n\nИстория диалога: {summary_history}\n\nОтвет:"}
    ]

    # Генерация ответа
    start_time = time.time()
    completion = model.create_chat_completion(
        max_tokens=max_tokens,
        temperature=temp,
        messages=messages
    )
    end_time = time.time()

    elapsed_time = end_time - start_time
    print(f"⏳ Время генерации ответа: {elapsed_time:.2f} секунд")

    # Получение ответа модели
    answer = completion['choices'][0]['message']['content']
    formatted_answer = textwrap.fill(answer, width=120)

    return formatted_answer

In [ ]:
# Задаем температуру генерации ответа
# Температура 0 означает, что модель будет давать более детерминированные ответы, строго опираясь на предоставленный контекст
temperature = 0

# Задаем количество релевантных чанков (фрагментов текста) для формирования ответа
relevant_chanks = 3

# Флаг для включения/выключения вывода релевантных чанков
#если 1 - выводятся чанки из базы знаний, которые подаются в модель, + ответ модели,  если 0 - только ответ модели
verbose = 1

# Имя модели, используемой для генерации ответов
model = llm

max_tokens=1000 # Количество токенов на ответ

# История диалога
history = [""]

In [ ]:
#Системный промпт
system = '''# Ты профессиональный помощник врача в чате с пациентом

## 1. Общие обязанности и цели
- У вас есть большой медицинский документ со сведениями о симптомах и диагнозах.
- Ваша обязанность: задавать пациенту краткие и корректные вопросы на русском языке, опираясь только на отрывки из этого документа.
- Ваша цель: уточнять симптомы пациента максимально точно и кратко, не добавляя ничего от себя.
- Пациент будет сообщать вам свои симптомы, а вам нужно задать 3 дополнительных вопроса, основанных на информации из документа о сопутствующих симптомах.

## 2. Ограничения в общении
- Запрещено отвечать на любые вопросы, не связанные с уточнением симптомов и сбором анамнеза.
- Если пациент спрашивает о чём-либо, не относящемся к симптомам, вам следует отказаться от ответа или напомнить пользователю для чего вы здесь.
- Нельзя предлагать варианты лечения или ставить диагноз.
- Нельзя рассказывать пользователю о его диагнозе.

## 3. Секретность документа
- Запрещено упоминать, что вы пользуетесь каким-либо справочником или дополнительными источниками информации.
- Ответы должны выглядеть так, будто они сформированы исключительно из ваших знаний и опыта.
'''

# Инструкция для модели о том, как использовать контекст для ответа на вопрос
instruction = '''Проанализируй предыдущий диалог чтобы написать свой ответ последовательным и логичным.
Категорически запрещено повторяться и здороваться.
Используйте следующие фрагменты контекста, чтобы ответить на вопрос в конце.
Стилистика ответа должна быть поддерживающей беседу в контексте уточнения симптомов.
Если вы не знаете ответа, просто скажите, что не знаете, не пытайтесь придумывать ответ.
Тебе запрещено сообщать диагноз. Запрещено спрашивать клиента что его еще интересует.'''

Bot

In [ ]:
import nest_asyncio
nest_asyncio.apply()

# Хранилище данных пользователей
user_histories = {}
user_questions_count = {}

# Клавиатура с кнопками
reply_keyboard = [["🔄 Начать новый чат", "✅ Записать симптомы"]]
markup = ReplyKeyboardMarkup(reply_keyboard, one_time_keyboard=False, resize_keyboard=True)

async def start(update: Update, context: CallbackContext) -> None:
    """Начало диалога."""
    user_id = update.message.chat_id
    user_histories[user_id] = []
    user_questions_count[user_id] = 0
    await update.message.reply_text("👋 Здравствуйте! Пожалуйста, опишите ваши симптомы.", reply_markup=markup)

async def handle_message(update: Update, context: CallbackContext) -> None:
    """Обработка сообщений пользователя."""
    user_id = update.message.chat_id
    user_message = update.message.text

    if user_message == "🔄 Начать новый диалог":
        await reset(update, context)
        return
    elif user_message == "✅ Записать симптомы":
        await save_symptoms(update, context)
        return

    user_histories[user_id].append(user_message)

    if user_questions_count[user_id] < 3:
        follow_up_question = answer_kons_llama(
            llm, system, instruction, user_message, faiss_index,
            user_histories[user_id], temp=temperature, verbose, k=relevant_chanks, max_tokens
        )
        user_questions_count[user_id] += 1
        await update.message.reply_text(follow_up_question, reply_markup=markup)
    else:
        await save_symptoms(update, context)

async def save_symptoms(update: Update, context: CallbackContext) -> None:
    """Фиксирует симптомы и завершает диалог."""
    user_id = update.message.chat_id
    final_summary = "\n".join(user_histories[user_id])
    await update.message.reply_text(
        f"✅ Спасибо за информацию! Ваши симптомы зафиксированы:\n\n{final_summary}",
        reply_markup=markup
    )
    user_histories[user_id] = []
    user_questions_count[user_id] = 0

async def reset(update: Update, context: CallbackContext) -> None:
    """Сбрасывает историю общения."""
    user_id = update.message.chat_id
    user_histories[user_id] = []
    user_questions_count[user_id] = 0
    await update.message.reply_text("🔄 История очищена.", reply_markup=markup)

async def main() -> None:
    """Запуск Telegram-бота. Добавьте вой токен"""
    app = Application.builder().token("....Здесь должен быть токен вашего бота....").build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    await app.run_polling()

In [ ]:
await main()

Выбранные чанки:
 
--------------------
- ощущение переполнения или дискомфорта в животе;
- боль в правом плече.

 
--------------------
чувство першения, дискомфорта и болезненности в глотке, покашливание преимущественно в утренние часы.

 
--------------------
Основные симптомы гнойного воспаления периоста – это болевые ощущения и отечность со стороны пораженной щеки. При развитии гнойного процесса в периостальной зоне нарушается общее самочувствие, появляется головная боль, слабость и незначительный подъем температуры тела. Пациент замечает, что до появления этих симптомов, у него болел зуб, при этом медицинская помощь не была получена. Если воспаляется нижняя челюсть со стороны язычной области, то человека беспокоят боли при употреблении пищи и глотании.

⏳ Время генерации ответа: 169.27 секунд
Выбранные чанки:
 
--------------------
Эти три симптома – интенсивные боли в правом подреберье, желтуха, лихорадка – являются ведущими признаками острого холангита, носят название «триада Ш

RuntimeError: Cannot close a running event loop